# Loan Default Risk Dashboard

An interactive dashboard for the loan default prediction model, built with
Panel and Bokeh. Designed in three layers:

1. **Overview** — plain-language summary for a general audience
2. **Look Up an Applicant** — interactive per-applicant risk explanation, with an optional technical toggle
3. **Model Details** — full technical internals (LASSO coefficients, global SHAP importance)

Requires `loan_data.csv` in the same folder (see `lasso_misclassification.py` for how to get it).


In [11]:
import numpy as np
import pandas as pd
import panel as pn
import shap
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, HoverTool
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

pn.extension(comms='vscode', theme='default')

## Load data and define the model pipeline

In [12]:
def load_and_prep(filepath):
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.replace(".", "_", regex=False)
    df['purpose'] = df['purpose'].astype('category')
    return df

NUMERIC_FEATURES = ['int_rate', 'installment', 'log_annual_inc', 'dti', 'fico',
                     'days_with_cr_line', 'revol_bal', 'revol_util',
                     'inq_last_6mths', 'delinq_2yrs', 'pub_rec']
CATEGORICAL_FEATURES = ['purpose']

def build_pipeline():
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), NUMERIC_FEATURES),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CATEGORICAL_FEATURES),
    ])
    model = LogisticRegressionCV(
        l1_ratios=[1.0], solver='liblinear', cv=5,
        scoring='roc_auc', max_iter=5000, random_state=42,
        class_weight='balanced'
    )
    return Pipeline([('preprocess', preprocessor), ('model', model)])

df = load_and_prep("loan_data.csv")
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df['not_fully_paid']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

pipeline = build_pipeline()
pipeline.fit(X_train, y_train)

X_test_t = pipeline.named_steps['preprocess'].transform(X_test)
if hasattr(X_test_t, "toarray"):
    X_test_t = X_test_t.toarray()

feature_names = (NUMERIC_FEATURES +
                  list(pipeline.named_steps['preprocess']
                       .named_transformers_['cat']
                       .get_feature_names_out(CATEGORICAL_FEATURES)))

masker = shap.maskers.Independent(X_test_t, max_samples=len(X_test_t))
explainer = shap.LinearExplainer(pipeline.named_steps['model'], masker)
shap_values = explainer(X_test_t)
proba = pipeline.named_steps['model'].predict_proba(X_test_t)[:, 1]

X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

print(f"Trained on {len(X_train)} applicants, evaluating on {len(X_test)}")

/Users/aprilren/Loan-Default-Risk-Prediction/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:2150: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


Trained on 7183 applicants, evaluating on 2395


## Plain-language explanation layer

Translates raw feature names and SHAP values into sentences a non-technical reader can follow.

In [13]:
FEATURE_INFO = {
    'fico': ("credit score", "A lower credit score generally means higher risk."),
    'dti': ("debt-to-income ratio", "A higher ratio means more of their income is already committed to debt."),
    'int_rate': ("interest rate offered", "Higher rates are often assigned to riskier borrowers to begin with."),
    'installment': ("monthly payment amount", "Larger monthly payments can be harder to sustain."),
    'log_annual_inc': ("annual income", "Lower income can mean less cushion to absorb a missed payment."),
    'inq_last_6mths': ("recent credit inquiries", "Many recent inquiries can signal financial stress or credit-seeking."),
    'revol_bal': ("revolving credit balance", "A higher balance means more existing debt outstanding."),
    'revol_util': ("credit utilization", "Using a high percentage of available credit increases risk."),
    'pub_rec': ("derogatory public records", "Records like bankruptcies increase risk substantially."),
    'delinq_2yrs': ("past delinquencies", "A history of late payments is a strong risk signal."),
    'days_with_cr_line': ("credit history length", "A longer credit history generally lowers risk."),
    'purpose_debt_consolidation': ("loan purpose: debt consolidation", ""),
    'purpose_credit_card': ("loan purpose: credit card", ""),
    'purpose_small_business': ("loan purpose: small business", "Small business loans carry structurally higher risk."),
    'purpose_home_improvement': ("loan purpose: home improvement", ""),
    'purpose_major_purchase': ("loan purpose: major purchase", ""),
}

def plain_language_summary(proba, top_factors):
    risk_word = "high" if proba > 0.6 else "moderate" if proba > 0.35 else "low"
    lines = [f"**Estimated default risk: {proba:.0%} ({risk_word})**", ""]

    increasing = [f for f in top_factors if f[1] > 0]
    decreasing = [f for f in top_factors if f[1] < 0]

    if increasing:
        lines.append("**Factors pushing risk up:**")
        for name, val in increasing[:3]:
            friendly_name, explanation = FEATURE_INFO.get(name, (name, ""))
            lines.append(f"- {friendly_name.capitalize()}. {explanation}")
    if decreasing:
        lines.append("")
        lines.append("**Factors pushing risk down:**")
        for name, val in decreasing[:2]:
            friendly_name, explanation = FEATURE_INFO.get(name, (name, ""))
            lines.append(f"- {friendly_name.capitalize()}.")

    return "\n".join(lines)

## Tab 1: Overview

In [14]:
overview_intro = pn.pane.Markdown(f"""
# Loan Default Risk Model

This model estimates the chance that a loan applicant will **not fully repay**
their loan, based on factors like credit score, income, and existing debt.

Out of **{len(df)}** historical applicants in this dataset,
**{y.mean():.0%}** eventually defaulted.

The model is tuned to prioritize **catching real defaults**, even if that means
occasionally flagging a safe borrower as risky. In lending, missing a real
default is usually more costly than being overly cautious about a good one.
""")

hist_vals, edges = np.histogram(proba, bins=25, range=(0, 1))
risk_hist = figure(height=280, width=500, title="Distribution of predicted risk across applicants",
                    x_axis_label="Predicted default risk", y_axis_label="Number of applicants",
                    toolbar_location=None)
risk_hist.quad(top=hist_vals, bottom=0, left=edges[:-1], right=edges[1:],
                fill_color="#4C78A8", line_color="white")

technical_metrics = pn.pane.Markdown(f"""
**ROC AUC:** {roc_auc_score(y_test, proba):.3f}

```
{classification_report(y_test, (proba > 0.5).astype(int))}
```
""")

overview_tab = pn.Column(
    overview_intro,
    pn.pane.Bokeh(risk_hist),
    pn.Accordion(("Show technical performance metrics", technical_metrics), active=[]),
)

## Tab 2: Look Up an Applicant

In [15]:
applicant_selector = pn.widgets.IntSlider(name="Applicant #", start=0, end=len(X_test_reset) - 1, value=0)
show_technical = pn.widgets.Checkbox(name="Show technical details (raw SHAP values)", value=False)

@pn.depends(applicant_selector.param.value, show_technical.param.value)
def applicant_view(idx, show_tech):
    contributions = sorted(zip(feature_names, shap_values.values[idx]), key=lambda x: -abs(x[1]))
    summary_pane = pn.pane.Markdown(plain_language_summary(proba[idx], contributions))

    if not show_tech:
        return pn.Column(summary_pane)

    names = [c[0] for c in contributions[:8]]
    vals = [c[1] for c in contributions[:8]]
    colors = ["#d62728" if v > 0 else "#2ca02c" for v in vals]
    src = ColumnDataSource(data={'name': names, 'value': vals, 'color': colors})
    p = figure(y_range=names[::-1], height=300, width=500, toolbar_location=None,
               title="SHAP contributions (exact values) for this applicant")
    p.hbar(y='name', right='value', height=0.6, color='color', source=src)
    p.add_tools(HoverTool(tooltips=[("feature", "@name"), ("SHAP value", "@value{0.000}")]))

    raw_data = pn.pane.DataFrame(X_test_reset.iloc[[idx]].T.rename(columns={idx: "value"}), width=400)

    return pn.Column(summary_pane, pn.pane.Bokeh(p), pn.pane.Markdown("**Raw applicant data:**"), raw_data)

applicant_tab = pn.Column(
    pn.pane.Markdown("## Look up an individual applicant"),
    applicant_selector, show_technical, applicant_view
)

## Tab 3: Model Details (technical)

In [16]:
global_importance = np.abs(shap_values.values).mean(axis=0)
top_global = sorted(zip(feature_names, global_importance), key=lambda x: -x[1])[:10]
src2 = ColumnDataSource(data={'name': [t[0] for t in top_global][::-1], 'value': [t[1] for t in top_global][::-1]})
p2 = figure(y_range=[t[0] for t in top_global][::-1], height=350, width=500, toolbar_location=None,
            title="Global feature importance (mean |SHAP value|)")
p2.hbar(y='name', right='value', height=0.6, color="#4C78A8", source=src2)

model_details_tab = pn.Column(
    pn.pane.Markdown("## Model internals"),
    pn.pane.Markdown("LASSO-regularized logistic regression, class_weight set to balanced "
                      "to address the class imbalance in the training data (most applicants "
                      "do NOT default, so the model is explicitly told missing a real default "
                      "is a costlier mistake than flagging a safe borrower)."),
    pn.pane.Bokeh(p2),
)

## Assemble and display the dashboard

In [10]:
dashboard = pn.Tabs(
    ("Overview", overview_tab),
    ("Look Up an Applicant", applicant_tab),
    ("Model Details", model_details_tab),
)

dashboard.servable()
dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'9ea6ab72-f14f-4ca5-bb8f-08b7f5c50e05': {'version…